# 07C — Tanore ↔ Manda Cross-Area Leave-One-Date-Out Ablation
## Correct Reference-Point Aggregation using `source_feature`

Diagnostic inspection confirmed that:

- `sample_id` identifies individual extracted pixel rows;
- `source_feature` identifies the original reference point.

Expected unique `source_feature` counts:
- Tanore Training = 289
- Tanore Validation = 97
- Manda Training = 291
- Manda Validation = 96

Therefore, all point-level thresholding, target evaluation, bootstrap summaries, and McNemar tests in this notebook use **`source_feature` as the reference-point identifier**.

The model is still trained on extracted pixels using grouped spatial CV, but independent accuracy statistics are calculated after averaging probabilities to one value per original reference point.

### GitHub execution note
This notebook preserves the publication analysis logic. Local absolute paths were replaced with the portable `BORO_PROJECT_ROOT` setting. Run Jupyter from the repository root or set that environment variable before execution. Generated figures and tables are written below `Outputs/`; licensed source imagery is not included.


In [ ]:

# CELL 1 — Imports, paths, and settings

from pathlib import Path
import json
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy.stats import binomtest

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedGroupKFold,
    cross_val_predict,
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(
    os.environ.get(
        "BORO_PROJECT_ROOT",
        str(Path.cwd()),
    )
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "Outputs"
    / "Q1_Extensions"
    / "Cross_Area_Leave_One_Date_Out_Ablation_SourceFeature_PointLevel"
)

TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"

for folder in [OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
CV_SPLITS = 5

# Final publication run
RF_SEARCH_ITERATIONS = 15
XGB_SEARCH_ITERATIONS = 20
BOOTSTRAP_REPLICATES = 1000

# True = fast diagnostic only
# False = final publication run
QUICK_MODE = False

if QUICK_MODE:
    RF_SEARCH_ITERATIONS = 3
    XGB_SEARCH_ITERATIONS = 3
    BOOTSTRAP_REPLICATES = 200

AREAS = ["Tanore", "Manda"]
STREAMS = ["FusedHybrid", "PlanetOnly"]
MODELS = ["RandomForest", "XGBoost"]

TRANSFER_DIRECTIONS = [
    ("Tanore", "Manda"),
    ("Manda", "Tanore"),
]

DATE_ORDER = ["Jan", "Mar", "Apr1", "Apr2"]
BAND_NAMES = ["Blue", "Green", "Red", "NIR"]

ABLATION_CONDITIONS = {
    "All_4_Dates": ["Jan", "Mar", "Apr1", "Apr2"],
    "Without_Jan": ["Mar", "Apr1", "Apr2"],
    "Without_Mar": ["Jan", "Apr1", "Apr2"],
    "Without_Apr1": ["Jan", "Mar", "Apr2"],
    "Without_Apr2": ["Jan", "Mar", "Apr1"],
}

print("Project root :", PROJECT_ROOT)
print("Output folder:", OUTPUT_ROOT)
print("Quick mode   :", QUICK_MODE)
print("Total experiments:", 2 * 2 * 2 * 5)


In [ ]:

# CELL 2 — Exact original classification feature schema

DIRECT_FEATURES = [
    feature
    for date_name in DATE_ORDER
    for feature in [
        *(f"{date_name}_{band}" for band in BAND_NAMES),
        f"{date_name}_NDVI",
    ]
]

SUMMARY_FEATURES = [
    "NDVI_mean",
    "NDVI_std",
    "NDVI_min",
    "NDVI_max",
    "NDVI_amplitude",
    "NDVI_peak_timing",
]

# These are the four temporal-difference features used by the main classification.
TEMPORAL_DIFF_DEFINITIONS = {
    "dNDVI_Mar_Jan": ("Jan", "Mar"),
    "dNDVI_Apr1_Mar": ("Mar", "Apr1"),
    "dNDVI_Apr2_Apr1": ("Apr1", "Apr2"),
    "dNDVI_Apr1_Jan": ("Jan", "Apr1"),
}

ORIGINAL_30_FEATURES = (
    DIRECT_FEATURES
    + [
        "NDVI_mean",
        "NDVI_std",
        "NDVI_min",
        "NDVI_max",
        "NDVI_amplitude",
        "dNDVI_Mar_Jan",
        "dNDVI_Apr1_Mar",
        "dNDVI_Apr2_Apr1",
        "dNDVI_Apr1_Jan",
        "NDVI_peak_timing",
    ]
)

assert len(ORIGINAL_30_FEATURES) == 30

print("Original full-date feature count:", len(ORIGINAL_30_FEATURES))
print(ORIGINAL_30_FEATURES)


In [ ]:
# CELL 3 — Load final tables and identify original reference points

REFERENCE_ID_COL = "source_feature"

def table_path(area, stream, split):
    return (
        PROJECT_ROOT
        / "Outputs"
        / area
        / "Classification_Q1"
        / "tables"
        / f"Q1_{stream}_{split}_Samples.csv"
    )


def load_table(area, stream, split):
    path = table_path(area, stream, split)

    if not path.exists():
        raise FileNotFoundError(
            "\nRequired file was not found:\n"
            f"{path}\n\nRun the final 03_{area}_Q1_Classification notebook first."
        )

    df = pd.read_csv(path)

    required = {
        "sample_id",
        REFERENCE_ID_COL,
        "class",
        "group",
        *DIRECT_FEATURES,
    }

    missing = sorted(required - set(df.columns))

    if missing:
        raise ValueError(
            f"{path.name} is missing required columns:\n{missing}"
        )

    if df[DIRECT_FEATURES].isna().any().any():
        bad_rows = int(
            df[DIRECT_FEATURES]
            .isna()
            .any(axis=1)
            .sum()
        )
        raise ValueError(
            f"{path.name} contains {bad_rows} rows with missing predictors."
        )

    classes = set(
        df["class"]
        .dropna()
        .astype(int)
        .unique()
    )

    if classes != {0, 1}:
        raise ValueError(
            f"{path.name}: class must contain both 0 and 1. Found {classes}"
        )

    # One original reference point must have exactly one class.
    label_counts = (
        df.groupby(REFERENCE_ID_COL)["class"]
        .nunique()
    )

    if (label_counts > 1).any():
        bad_ids = (
            label_counts[
                label_counts > 1
            ]
            .index
            .tolist()
        )
        raise ValueError(
            f"{path.name}: source_feature has conflicting labels: {bad_ids[:10]}"
        )

    return df.copy()


sample_tables = {
    area: {
        stream: {
            split: load_table(area, stream, split)
            for split in ["Training", "Validation"]
        }
        for stream in STREAMS
    }
    for area in AREAS
}

summary_rows = []

for area in AREAS:
    for stream in STREAMS:
        for split in ["Training", "Validation"]:
            df = sample_tables[area][stream][split]

            point_labels = (
                df[
                    [
                        REFERENCE_ID_COL,
                        "class",
                    ]
                ]
                .drop_duplicates(
                    REFERENCE_ID_COL
                )
            )

            summary_rows.append({
                "area": area,
                "stream": stream,
                "split": split,
                "n_pixel_rows": len(df),
                "n_reference_points": int(
                    df[REFERENCE_ID_COL].nunique()
                ),
                "n_rice_reference_points": int(
                    (point_labels["class"] == 1).sum()
                ),
                "n_nonrice_reference_points": int(
                    (point_labels["class"] == 0).sum()
                ),
                "n_spatial_groups": int(
                    df["group"]
                    .astype(str)
                    .nunique()
                ),
            })

input_summary = pd.DataFrame(
    summary_rows
)

display(input_summary)

expected_counts = {
    ("Tanore", "Training"): 289,
    ("Tanore", "Validation"): 97,
    ("Manda", "Training"): 291,
    ("Manda", "Validation"): 96,
}

for (area, split), expected in expected_counts.items():
    observed = int(
        input_summary[
            (input_summary["area"] == area)
            & (input_summary["split"] == split)
        ]["n_reference_points"]
        .iloc[0]
    )

    if observed != expected:
        raise ValueError(
            f"{area} {split}: expected {expected} original reference points "
            f"from source_feature, but found {observed}."
        )

print("\n✅ `source_feature` correctly recovers the original reference points.")
print("✅ Expected counts verified: 289 / 97 / 291 / 96.")

In [ ]:

# CELL 4 — Rebuild features after removing a date

DATE_POSITION = {
    "Jan": 0.0,
    "Mar": 1.0,
    "Apr1": 2.0,
    "Apr2": 3.0,
}


def build_ablation_features(df, active_dates):
    """
    Publication-safe feature rebuilding.

    Full-date case exactly reproduces the original 30-feature schema.
    For an omitted date:
      - its direct bands + NDVI are removed;
      - NDVI summaries are recomputed from remaining dates;
      - NDVI peak timing is recomputed from remaining dates;
      - a temporal difference is retained only when BOTH endpoint dates remain.
    """

    active_dates = list(active_dates)

    feature_frame = pd.DataFrame(index=df.index)
    feature_names = []

    # Direct spectral + NDVI features.
    for date_name in active_dates:
        cols = [
            *[f"{date_name}_{band}" for band in BAND_NAMES],
            f"{date_name}_NDVI",
        ]

        for col in cols:
            feature_frame[col] = df[col].astype("float32")

        feature_names.extend(cols)

    # Recompute NDVI summary features from active dates only.
    ndvi_cols = [
        f"{date_name}_NDVI"
        for date_name in active_dates
    ]

    ndvi = df[ndvi_cols].to_numpy(dtype="float32")

    feature_frame["NDVI_mean"] = np.mean(ndvi, axis=1)
    feature_frame["NDVI_std"] = np.std(ndvi, axis=1, ddof=0)
    feature_frame["NDVI_min"] = np.min(ndvi, axis=1)
    feature_frame["NDVI_max"] = np.max(ndvi, axis=1)
    feature_frame["NDVI_amplitude"] = (
        feature_frame["NDVI_max"]
        - feature_frame["NDVI_min"]
    )

    peak_local_idx = np.argmax(ndvi, axis=1)

    time_positions = np.array(
        [DATE_POSITION[d] for d in active_dates],
        dtype="float32",
    )

    feature_frame["NDVI_peak_timing"] = (
        time_positions[peak_local_idx]
    )

    feature_names.extend([
        "NDVI_mean",
        "NDVI_std",
        "NDVI_min",
        "NDVI_max",
        "NDVI_amplitude",
    ])

    # Keep only original temporal differences whose endpoint dates remain.
    for feature_name, (earlier, later) in TEMPORAL_DIFF_DEFINITIONS.items():

        if earlier in active_dates and later in active_dates:
            feature_frame[feature_name] = (
                df[f"{later}_NDVI"]
                - df[f"{earlier}_NDVI"]
            ).astype("float32")

            feature_names.append(feature_name)

    feature_names.append("NDVI_peak_timing")

    # Reorder full-date case to the exact original 30-feature schema.
    if active_dates == DATE_ORDER:
        missing_full = [
            f for f in ORIGINAL_30_FEATURES
            if f not in feature_frame.columns
        ]

        if missing_full:
            raise RuntimeError(
                f"Full-date feature rebuilding failed. Missing: {missing_full}"
            )

        feature_names = ORIGINAL_30_FEATURES.copy()

    X = feature_frame[feature_names].to_numpy(dtype="float32")

    if not np.isfinite(X).all():
        raise ValueError(
            "Non-finite values found in rebuilt ablation predictors."
        )

    return X, feature_names


schema_rows = []

test_df = sample_tables["Tanore"]["PlanetOnly"]["Training"]

for condition, active_dates in ABLATION_CONDITIONS.items():
    _, names = build_ablation_features(
        test_df,
        active_dates,
    )

    schema_rows.append({
        "condition": condition,
        "active_dates": ", ".join(active_dates),
        "feature_count": len(names),
        "features": ", ".join(names),
    })

feature_schema = pd.DataFrame(schema_rows)

display(
    feature_schema[
        ["condition", "active_dates", "feature_count"]
    ]
)

assert int(
    feature_schema.loc[
        feature_schema["condition"] == "All_4_Dates",
        "feature_count",
    ].iloc[0]
) == 30

print("\n✅ Full-date baseline = exact original 30-feature schema.")
print("✅ Removed-date information is excluded from ablation features.")


In [ ]:
# CELL 5 — CV, point aggregation, model search, threshold, metrics, bootstrap

def make_spatial_cv(y, groups):
    group_table = (
        pd.DataFrame({"group": groups, "class": y})
        .drop_duplicates()
    )

    class_group_counts = (
        group_table.groupby("class")["group"].nunique()
    )

    n_splits = min(
        CV_SPLITS,
        int(class_group_counts.min()),
        int(group_table["group"].nunique()),
    )

    if n_splits < 3:
        raise ValueError(
            "At least 3 independent source-area groups per class are required."
        )

    return StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_SEED,
    )


def aggregate_to_reference_points(
    reference_ids,
    y_true,
    groups,
    probabilities,
):
    frame = pd.DataFrame({
        "reference_id": pd.Series(reference_ids).astype(str).to_numpy(),
        "true_class": np.asarray(y_true, dtype=int),
        "group": pd.Series(groups).astype(str).to_numpy(),
        "probability": np.asarray(probabilities, dtype=float),
    })

    label_n = frame.groupby("reference_id")["true_class"].nunique()
    if (label_n > 1).any():
        raise ValueError(
            "At least one sample_id contains conflicting reference labels."
        )

    group_n = frame.groupby("reference_id")["group"].nunique()
    if (group_n > 1).any():
        print(
            "⚠ Some reference_id values span >1 spatial group; first group retained."
        )

    point = (
        frame
        .groupby("reference_id", as_index=False)
        .agg(
            true_class=("true_class", "first"),
            group=("group", "first"),
            probability=("probability", "mean"),
            n_pixel_rows=("probability", "size"),
        )
    )

    return point


def optimal_f1_threshold(y_true, probability):
    precision, recall, thresholds = precision_recall_curve(
        y_true,
        probability,
    )

    if thresholds.size == 0:
        return 0.5

    f1_values = (
        2.0 * precision[:-1] * recall[:-1]
        / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    )

    return float(
        thresholds[int(np.nanargmax(f1_values))]
    )


def make_estimator_and_search_space(model_name, y):
    if model_name == "RandomForest":

        estimator = RandomForestClassifier(
            random_state=RANDOM_SEED,
            class_weight="balanced",
            n_jobs=1,
        )

        search_space = {
            "n_estimators": [300, 500, 800, 1000],
            "max_depth": [None, 10, 15, 20, 30],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4, 8],
            "max_features": ["sqrt", "log2", 0.4, 0.7],
            "bootstrap": [True, False],
        }

        if QUICK_MODE:
            search_space = {
                "n_estimators": [80, 120, 180],
                "max_depth": [None, 10, 15],
                "min_samples_split": [2, 5],
                "min_samples_leaf": [1, 2, 4],
                "max_features": ["sqrt", 0.7],
                "bootstrap": [True, False],
            }

        n_iter = RF_SEARCH_ITERATIONS

    elif model_name == "XGBoost":

        negative = max(int((y == 0).sum()), 1)
        positive = max(int((y == 1).sum()), 1)

        estimator = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=1,
            scale_pos_weight=negative / positive,
        )

        search_space = {
            "n_estimators": [300, 500, 700, 1000],
            "max_depth": [3, 4, 5, 6, 8],
            "learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1],
            "subsample": [0.65, 0.8, 0.9, 1.0],
            "colsample_bytree": [0.6, 0.75, 0.9, 1.0],
            "min_child_weight": [1, 3, 5, 8],
            "gamma": [0.0, 0.1, 0.3],
            "reg_alpha": [0.0, 0.01, 0.1, 0.5],
            "reg_lambda": [0.5, 1.0, 2.0, 5.0],
        }

        if QUICK_MODE:
            search_space = {
                "n_estimators": [80, 120, 180],
                "max_depth": [3, 4, 5],
                "learning_rate": [0.05, 0.1],
                "subsample": [0.8, 1.0],
                "colsample_bytree": [0.75, 1.0],
                "min_child_weight": [1, 3],
                "gamma": [0.0, 0.1],
                "reg_alpha": [0.0, 0.1],
                "reg_lambda": [1.0, 2.0],
            }

        n_iter = XGB_SEARCH_ITERATIONS

    else:
        raise ValueError(f"Unknown model: {model_name}")

    return estimator, search_space, n_iter


def tune_source_full_date(
    model_name,
    X_source_full,
    y_source,
    groups_source,
):
    cv = make_spatial_cv(y_source, groups_source)

    estimator, search_space, n_iter = (
        make_estimator_and_search_space(
            model_name,
            y_source,
        )
    )

    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=search_space,
        n_iter=n_iter,
        scoring="average_precision",
        n_jobs=-1,
        cv=cv,
        random_state=RANDOM_SEED,
        refit=True,
        verbose=1,
    )

    search.fit(
        X_source_full,
        y_source,
        groups=groups_source,
    )

    return {
        "best_params": search.best_params_,
        "best_cv_ap": float(search.best_score_),
    }


def build_frozen_estimator(
    model_name,
    y_source,
    best_params,
):
    estimator, _, _ = make_estimator_and_search_space(
        model_name,
        y_source,
    )
    estimator.set_params(**best_params)
    return estimator


def train_source_condition(
    model_name,
    X_source,
    y_source,
    groups_source,
    reference_ids_source,
    frozen_params,
):
    cv = make_spatial_cv(
        y_source,
        groups_source,
    )

    estimator = build_frozen_estimator(
        model_name,
        y_source,
        frozen_params,
    )

    oof_probability_pixels = cross_val_predict(
        estimator,
        X_source,
        y_source,
        groups=groups_source,
        cv=cv,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]

    source_oof_points = aggregate_to_reference_points(
        reference_ids_source,
        y_source,
        groups_source,
        oof_probability_pixels,
    )

    threshold = optimal_f1_threshold(
        source_oof_points["true_class"].to_numpy(dtype=int),
        source_oof_points["probability"].to_numpy(dtype=float),
    )

    final_model = clone(estimator)
    final_model.fit(X_source, y_source)

    return final_model, threshold, source_oof_points


def metric_row(y_true, probability, prediction):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        prediction,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp)
        else np.nan
    )

    return {
        "n_reference_points": int(len(y_true)),
        "OA": accuracy_score(y_true, prediction),
        "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "specificity": specificity,
        "F1": f1_score(y_true, prediction, zero_division=0),
        "kappa": cohen_kappa_score(y_true, prediction),
        "MCC": matthews_corrcoef(y_true, prediction),
        "ROC_AUC": roc_auc_score(y_true, probability),
        "PR_AUC": average_precision_score(y_true, probability),
        "Brier": brier_score_loss(y_true, probability),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }


def cluster_bootstrap_f1(
    point_frame,
    replicates,
    seed,
):
    frame = point_frame.copy().reset_index(drop=True)

    unique_groups = (
        frame["group"]
        .astype(str)
        .drop_duplicates()
        .to_numpy()
    )

    rng = np.random.default_rng(seed)
    values = []

    for _ in range(replicates):
        chosen_groups = rng.choice(
            unique_groups,
            size=len(unique_groups),
            replace=True,
        )

        sampled = pd.concat(
            [
                frame[
                    frame["group"].astype(str)
                    == str(group)
                ]
                for group in chosen_groups
            ],
            ignore_index=True,
        )

        values.append(
            f1_score(
                sampled["true_class"],
                sampled["prediction"],
                zero_division=0,
            )
        )

    return tuple(
        np.quantile(values, [0.025, 0.975])
    )


## Tuning logic

প্রতি source area + stream + model-এর জন্য full 4-date training set-এ tuning হবে:

- Random Forest: **15 candidates × 5 folds = 75 fits**
- XGBoost: **20 candidates × 5 folds = 100 fits**

তারপর best hyperparameters freeze থাকবে।  
প্রতিটি date-ablation condition-এ source training data দিয়ে OOF threshold নতুন করে নির্ধারণ হবে, কিন্তু target validation data tuning-এ ঢুকবে না।


In [ ]:

# CELL 6 — Tune full-date source models ONCE

frozen_models = {}
tuning_rows = []

for source_area, target_area in TRANSFER_DIRECTIONS:

    frozen_models[source_area] = {}

    for stream in STREAMS:

        frozen_models[source_area][stream] = {}

        source_train = (
            sample_tables[source_area]
            [stream]["Training"]
        )

        X_source_full, full_names = build_ablation_features(
            source_train,
            DATE_ORDER,
        )

        assert full_names == ORIGINAL_30_FEATURES

        y_source = (
            source_train["class"]
            .to_numpy(dtype=int)
        )

        groups_source = (
            source_train["group"]
            .astype(str)
            .to_numpy()
        )

        for model_name in MODELS:

            print("\n" + "=" * 78)
            print(
                f"FULL-DATE SOURCE TUNING: "
                f"{source_area} → {target_area} | "
                f"{stream} | {model_name}"
            )
            print("=" * 78)

            tuning_result = tune_source_full_date(
                model_name,
                X_source_full,
                y_source,
                groups_source,
            )

            frozen_models[
                source_area
            ][stream][model_name] = tuning_result

            tuning_rows.append({
                "source_area": source_area,
                "target_area": target_area,
                "stream": stream,
                "model": model_name,
                "feature_count": len(full_names),
                "best_cv_average_precision": tuning_result["best_cv_ap"],
                "best_parameters": json.dumps(
                    tuning_result["best_params"]
                ),
            })

            print(
                "Best source CV AP:",
                round(
                    tuning_result["best_cv_ap"],
                    6,
                ),
            )

tuning_table = pd.DataFrame(tuning_rows)

display(tuning_table)

print("\n✅ Source-only full-date tuning complete.")


In [ ]:
# CELL 7 — Run all 40 cross-area ablation experiments at REFERENCE-POINT LEVEL

result_rows = []
prediction_frames = []

for source_area, target_area in TRANSFER_DIRECTIONS:

    for stream in STREAMS:

        source_train = (
            sample_tables[source_area]
            [stream]["Training"]
            .copy()
        )

        target_valid = (
            sample_tables[target_area]
            [stream]["Validation"]
            .copy()
        )

        y_source = source_train["class"].to_numpy(dtype=int)
        groups_source = source_train["group"].astype(str).to_numpy()
        reference_ids_source = source_train[REFERENCE_ID_COL].astype(str).to_numpy()

        y_target_pixels = target_valid["class"].to_numpy(dtype=int)
        groups_target_pixels = target_valid["group"].astype(str).to_numpy()
        reference_ids_target = target_valid[REFERENCE_ID_COL].astype(str).to_numpy()

        for condition, active_dates in ABLATION_CONDITIONS.items():

            X_source, feature_names = build_ablation_features(
                source_train,
                active_dates,
            )

            X_target, target_feature_names = build_ablation_features(
                target_valid,
                active_dates,
            )

            if feature_names != target_feature_names:
                raise RuntimeError(
                    "Source/target feature schema mismatch."
                )

            for model_name in MODELS:

                print("\n" + "=" * 78)
                print(
                    f"{source_area} → {target_area} | "
                    f"{stream} | {model_name}"
                )
                print(f"CONDITION: {condition}")
                print(f"ACTIVE DATES: {active_dates}")
                print(f"FEATURES: {len(feature_names)}")
                print("=" * 78)

                frozen_params = (
                    frozen_models[
                        source_area
                    ][stream][model_name]["best_params"]
                )

                model, threshold, source_oof_points = (
                    train_source_condition(
                        model_name,
                        X_source,
                        y_source,
                        groups_source,
                        reference_ids_source,
                        frozen_params,
                    )
                )

                target_probability_pixels = (
                    model.predict_proba(X_target)[:, 1]
                )

                target_points = aggregate_to_reference_points(
                    reference_ids_target,
                    y_target_pixels,
                    groups_target_pixels,
                    target_probability_pixels,
                )

                target_points["prediction"] = (
                    target_points["probability"]
                    >= threshold
                ).astype("uint8")

                metrics = metric_row(
                    target_points["true_class"].to_numpy(dtype=int),
                    target_points["probability"].to_numpy(dtype=float),
                    target_points["prediction"].to_numpy(dtype=int),
                )

                ci_low, ci_high = cluster_bootstrap_f1(
                    target_points,
                    BOOTSTRAP_REPLICATES,
                    RANDOM_SEED,
                )

                date_removed = (
                    ""
                    if condition == "All_4_Dates"
                    else condition.replace("Without_", "")
                )

                result_rows.append({
                    "source_area": source_area,
                    "target_area": target_area,
                    "stream": stream,
                    "model": model_name,
                    "condition": condition,
                    "date_removed": date_removed,
                    "active_dates": ", ".join(active_dates),
                    "feature_count": len(feature_names),
                    "threshold_source_point_level": threshold,
                    "source_oof_reference_points": len(source_oof_points),
                    "target_pixel_rows": len(target_valid),
                    "F1_CI_low": ci_low,
                    "F1_CI_high": ci_high,
                    **metrics,
                })

                point_output = target_points.copy()
                point_output["source_area"] = source_area
                point_output["target_area"] = target_area
                point_output["stream"] = stream
                point_output["model"] = model_name
                point_output["condition"] = condition

                prediction_frames.append(point_output)

                print(
                    "Target reference points:",
                    len(target_points),
                    "| F1:",
                    round(metrics["F1"], 4),
                    "| MCC:",
                    round(metrics["MCC"], 4),
                    "| OA:",
                    round(metrics["OA"], 4),
                    "| threshold:",
                    round(threshold, 4),
                )

results = pd.DataFrame(result_rows)

predictions = pd.concat(
    prediction_frames,
    ignore_index=True,
)

print(
    "\n✅ ALL 40 POINT-LEVEL CROSS-AREA ABLATION EXPERIMENTS FINISHED"
)

In [ ]:

# CELL 8 — Calculate F1/MCC/OA drop relative to full 4-date transfer

baseline = (
    results[
        results["condition"] == "All_4_Dates"
    ][
        [
            "source_area",
            "target_area",
            "stream",
            "model",
            "F1",
            "MCC",
            "OA",
            "ROC_AUC",
            "PR_AUC",
            "Brier",
        ]
    ]
    .rename(
        columns={
            "F1": "baseline_F1",
            "MCC": "baseline_MCC",
            "OA": "baseline_OA",
            "ROC_AUC": "baseline_ROC_AUC",
            "PR_AUC": "baseline_PR_AUC",
            "Brier": "baseline_Brier",
        }
    )
)

ablation_results = results.merge(
    baseline,
    on=[
        "source_area",
        "target_area",
        "stream",
        "model",
    ],
    how="left",
)

ablation_results["F1_drop_vs_full"] = (
    ablation_results["baseline_F1"]
    - ablation_results["F1"]
)

ablation_results["MCC_drop_vs_full"] = (
    ablation_results["baseline_MCC"]
    - ablation_results["MCC"]
)

ablation_results["OA_drop_vs_full"] = (
    ablation_results["baseline_OA"]
    - ablation_results["OA"]
)

display(
    ablation_results[
        [
            "source_area",
            "target_area",
            "stream",
            "model",
            "condition",
            "F1",
            "baseline_F1",
            "F1_drop_vs_full",
            "MCC",
            "MCC_drop_vs_full",
            "OA",
            "OA_drop_vs_full",
        ]
    ].round(4)
)

print(
    "\nPositive F1_drop = removing that date reduced geographic transfer performance."
)
print(
    "Negative F1_drop = removing that date improved geographic transfer performance."
)


In [ ]:
# CELL 9 — McNemar exact test at REFERENCE-POINT LEVEL

mcnemar_rows = []

keys = [
    "source_area",
    "target_area",
    "stream",
    "model",
]

for key_values, group in predictions.groupby(keys):

    source_area, target_area, stream, model_name = key_values

    full = (
        group[
            group["condition"] == "All_4_Dates"
        ][
            ["reference_id", "true_class", "prediction"]
        ]
        .rename(
            columns={
                "prediction": "prediction_full"
            }
        )
    )

    for condition in [
        "Without_Jan",
        "Without_Mar",
        "Without_Apr1",
        "Without_Apr2",
    ]:

        omitted = (
            group[
                group["condition"] == condition
            ][
                ["reference_id", "true_class", "prediction"]
            ]
            .rename(
                columns={
                    "prediction": "prediction_omitted"
                }
            )
        )

        merged = full.merge(
            omitted,
            on=["reference_id", "true_class"],
            how="inner",
        )

        correct_full = (
            merged["prediction_full"]
            == merged["true_class"]
        )

        correct_omitted = (
            merged["prediction_omitted"]
            == merged["true_class"]
        )

        full_only_correct = int(
            (correct_full & ~correct_omitted).sum()
        )

        omitted_only_correct = int(
            (~correct_full & correct_omitted).sum()
        )

        discordant = (
            full_only_correct
            + omitted_only_correct
        )

        if discordant > 0:
            p_value = binomtest(
                full_only_correct,
                discordant,
                p=0.5,
                alternative="two-sided",
            ).pvalue
        else:
            p_value = 1.0

        mcnemar_rows.append({
            "source_area": source_area,
            "target_area": target_area,
            "stream": stream,
            "model": model_name,
            "condition": condition,
            "n_paired_reference_points": len(merged),
            "full_only_correct": full_only_correct,
            "omitted_only_correct": omitted_only_correct,
            "discordant": discordant,
            "mcnemar_exact_p": p_value,
        })

mcnemar_table = pd.DataFrame(mcnemar_rows)

display(mcnemar_table.round(4))

print(
    "\n✅ McNemar uses reference points, not extracted pixel rows."
)

In [ ]:

# CELL 10 — Date importance rankings

omitted = (
    ablation_results[
        ablation_results["date_removed"] != ""
    ]
    .copy()
)

# Overall across both directions, streams, and models.
overall_date_importance = (
    omitted
    .groupby(
        "date_removed",
        as_index=False,
    )
    .agg(
        mean_F1_drop=(
            "F1_drop_vs_full",
            "mean",
        ),
        median_F1_drop=(
            "F1_drop_vs_full",
            "median",
        ),
        mean_MCC_drop=(
            "MCC_drop_vs_full",
            "mean",
        ),
        mean_OA_drop=(
            "OA_drop_vs_full",
            "mean",
        ),
        min_F1_drop=(
            "F1_drop_vs_full",
            "min",
        ),
        max_F1_drop=(
            "F1_drop_vs_full",
            "max",
        ),
    )
    .sort_values(
        "mean_F1_drop",
        ascending=False,
    )
    .reset_index(drop=True)
)

overall_date_importance.insert(
    0,
    "importance_rank",
    np.arange(
        1,
        len(overall_date_importance) + 1,
    ),
)

# Direction-specific ranking.
direction_date_importance = (
    omitted
    .groupby(
        [
            "source_area",
            "target_area",
            "date_removed",
        ],
        as_index=False,
    )
    .agg(
        mean_F1_drop=(
            "F1_drop_vs_full",
            "mean",
        ),
        median_F1_drop=(
            "F1_drop_vs_full",
            "median",
        ),
        mean_MCC_drop=(
            "MCC_drop_vs_full",
            "mean",
        ),
        mean_OA_drop=(
            "OA_drop_vs_full",
            "mean",
        ),
    )
)

direction_date_importance["importance_rank_within_direction"] = (
    direction_date_importance
    .groupby(
        [
            "source_area",
            "target_area",
        ]
    )["mean_F1_drop"]
    .rank(
        method="dense",
        ascending=False,
    )
    .astype(int)
)

direction_date_importance = (
    direction_date_importance
    .sort_values(
        [
            "source_area",
            "target_area",
            "importance_rank_within_direction",
        ]
    )
)

print("OVERALL CROSS-AREA DATE IMPORTANCE")
display(
    overall_date_importance.round(4)
)

print("\nDIRECTION-SPECIFIC DATE IMPORTANCE")
display(
    direction_date_importance.round(4)
)


In [ ]:

# CELL 11 — Publication figures

condition_order = [
    "All_4_Dates",
    "Without_Jan",
    "Without_Mar",
    "Without_Apr1",
    "Without_Apr2",
]

for source_area, target_area in TRANSFER_DIRECTIONS:

    for stream in STREAMS:

        for model_name in MODELS:

            subset = (
                ablation_results[
                    (
                        ablation_results["source_area"]
                        == source_area
                    )
                    & (
                        ablation_results["target_area"]
                        == target_area
                    )
                    & (
                        ablation_results["stream"]
                        == stream
                    )
                    & (
                        ablation_results["model"]
                        == model_name
                    )
                ]
                .set_index("condition")
                .reindex(condition_order)
                .reset_index()
            )

            fig, ax = plt.subplots(
                figsize=(8.5, 5.0)
            )

            ax.bar(
                subset["condition"],
                subset["F1"],
            )

            ax.set_ylabel(
                "Independent target-area F1"
            )

            ax.set_title(
                f"{source_area} → {target_area} | "
                f"{stream} | {model_name}\n"
                "Cross-Area Leave-One-Date-Out Ablation"
            )

            ax.tick_params(
                axis="x",
                rotation=25,
            )

            minimum = float(
                subset["F1"].min()
            )

            ax.set_ylim(
                max(
                    0.0,
                    minimum - 0.08,
                ),
                1.01,
            )

            fig.tight_layout()

            fig.savefig(
                FIGURE_DIR
                / (
                    f"CrossArea_Ablation_F1_"
                    f"{source_area}_to_{target_area}_"
                    f"{stream}_{model_name}.png"
                ),
                dpi=300,
                bbox_inches="tight",
            )

            plt.close(fig)


# Overall date importance plot.
fig, ax = plt.subplots(
    figsize=(7.5, 4.8)
)

ax.bar(
    overall_date_importance["date_removed"],
    overall_date_importance["mean_F1_drop"],
)

ax.axhline(
    0.0,
    linewidth=1,
)

ax.set_xlabel(
    "Removed observation date"
)

ax.set_ylabel(
    "Mean cross-area F1 drop"
)

ax.set_title(
    "Overall Date Importance for Geographic Transfer"
)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "Overall_CrossArea_Date_Importance_Mean_F1_Drop.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("✅ Publication figures saved.")


In [ ]:

# CELL 12 — Save all results

input_summary.to_csv(
    TABLE_DIR
    / "CrossArea_Ablation_Input_Summary.csv",
    index=False,
)

feature_schema.to_csv(
    TABLE_DIR
    / "CrossArea_Ablation_Feature_Schema.csv",
    index=False,
)

tuning_table.to_csv(
    TABLE_DIR
    / "CrossArea_Ablation_Source_FullDate_Tuning.csv",
    index=False,
)

ablation_results.to_csv(
    TABLE_DIR
    / "CrossArea_Leave_One_Date_Out_Metrics.csv",
    index=False,
)

overall_date_importance.to_csv(
    TABLE_DIR
    / "CrossArea_Overall_Date_Importance.csv",
    index=False,
)

direction_date_importance.to_csv(
    TABLE_DIR
    / "CrossArea_Direction_Date_Importance.csv",
    index=False,
)

mcnemar_table.to_csv(
    TABLE_DIR
    / "CrossArea_Full_vs_Omitted_McNemar.csv",
    index=False,
)

predictions.to_csv(
    TABLE_DIR
    / "CrossArea_Ablation_Target_SourceFeature_Point_Predictions.csv",
    index=False,
)

excel_path = (
    OUTPUT_ROOT
    / "CrossArea_Leave_One_Date_Out_Ablation_SourceFeature_PointLevel_Results.xlsx"
)

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl",
) as writer:

    input_summary.to_excel(
        writer,
        sheet_name="Input_Summary",
        index=False,
    )

    feature_schema.to_excel(
        writer,
        sheet_name="Feature_Schema",
        index=False,
    )

    tuning_table.to_excel(
        writer,
        sheet_name="Source_Tuning",
        index=False,
    )

    ablation_results.to_excel(
        writer,
        sheet_name="Ablation_Metrics",
        index=False,
    )

    overall_date_importance.to_excel(
        writer,
        sheet_name="Overall_Date_Importance",
        index=False,
    )

    direction_date_importance.to_excel(
        writer,
        sheet_name="Direction_Importance",
        index=False,
    )

    mcnemar_table.to_excel(
        writer,
        sheet_name="Full_vs_Omitted_McNemar",
        index=False,
    )

    predictions.to_excel(
        writer,
        sheet_name="Target_Predictions",
        index=False,
    )

print("\n" + "=" * 78)
print("✅ POINT-LEVEL CROSS-AREA LEAVE-ONE-DATE-OUT ABLATION COMPLETE")
print("=" * 78)
print("Excel  :", excel_path)
print("Tables :", TABLE_DIR)
print("Figures:", FIGURE_DIR)


# Final checks

প্রথম input table-এ অবশ্যই দেখাবে:

- Tanore Training = **289 reference points**
- Tanore Validation = **97 reference points**
- Manda Training = **291 reference points**
- Manda Validation = **96 reference points**

এগুলো `source_feature` থেকে পাওয়া original reference points।

তারপর final results-এ:

- Tanore target হলে `n_reference_points = 97`
- Manda target হলে `n_reference_points = 96`

`F1_drop_vs_full`:
- Positive = date বাদ দিলে transfer performance কমেছে
- Negative = date বাদ দিলে transfer performance improve করেছে

Main output:
- `CrossArea_Overall_Date_Importance.csv`
- `CrossArea_Direction_Date_Importance.csv`
- `CrossArea_Full_vs_Omitted_McNemar.csv`
- `CrossArea_Ablation_Target_SourceFeature_Point_Predictions.csv`
- `CrossArea_Leave_One_Date_Out_Ablation_SourceFeature_PointLevel_Results.xlsx`

এই version-টাই final publication analysis হিসেবে ব্যবহার করবেন।